# **Import Library & Load Data**

In [1]:
import pandas as pd

In [ ]:
df = pd.read_parquet('../data/cleaned/ulasan_dana_cleaned.parquet')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 740000 entries, 0 to 739999
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   id_ulasan        740000 non-null  object        
 1   user             740000 non-null  object        
 2   ulasan           740000 non-null  object        
 3   rating           740000 non-null  int64         
 4   dukungan_ulasan  740000 non-null  int64         
 5   tanggal          740000 non-null  datetime64[ns]
 6   versi_aplikasi   740000 non-null  category      
dtypes: category(1), datetime64[ns](1), int64(2), object(3)
memory usage: 35.3+ MB


In [3]:
df.head()

,id_ulasan,user,ulasan,rating,dukungan_ulasan,tanggal,versi_aplikasi
0,8c382ad2-2c81-436e-a18e-87be3fdcc662,Hermansyah Radin,ok,5,0,2026-01-22 09:14:57,unknown
1,47553db5-9026-4233-8bf7-8449e550da69,Zidni Cell,bagus,5,0,2026-01-22 09:11:52,unknown
2,9a1a84ba-1b7d-4010-a58d-1f373b280082,Akhi Sondi,mantulll pokok eh,5,0,2026-01-22 09:11:39,2.110.0
3,0460a39e-7580-44bf-8a79-8446fb2414eb,Yati Yati,mempermudah,5,0,2026-01-22 09:09:57,2.110.0
4,b461e228-9033-4bea-b835-322d0db38fc2,Joker 16,mantap,5,0,2026-01-22 09:09:34,2.111.0


# **EDA**

Preparation
----

In [4]:
# %pip install plotly nbformat

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

template = "plotly_white"

Distribusi Rating dengan Persentase
----

In [6]:
rating_counts = df['rating'].value_counts().sort_index().reset_index()
rating_counts.columns = ['Rating', 'Jumlah']
total_data = len(df)
rating_counts['Persentase'] = (rating_counts['Jumlah'] / total_data * 100).round(2)

fig1 = px.bar(rating_counts, x='Rating', y='Jumlah', 
             text=rating_counts['Persentase'].apply(lambda x: f'{x}%'),
             color='Rating', color_continuous_scale='Viridis',
             title='<b>Distribusi Rating Aplikasi DANA</b>',
             template=template)
fig1.update_traces(textposition='outside')
fig1.show()

Tren Ulasan Berdasarkan Waktu (Interaktif)
----

In [ ]:
df_ts = df.groupby(df['tanggal'].dt.date).size().reset_index(name='Jumlah Ulasan')

fig2 = px.line(df_ts, x='tanggal', y='Jumlah Ulasan',
              title='<b>Tren Volume Ulasan Harian</b>',
              labels={'tanggal': 'Tanggal', 'Jumlah Ulasan': 'Total'},
              template=template)
fig2.update_traces(line_color='#1f77b4', line_width=2)
fig2.update_xaxes(rangeslider_visible=True)
fig2.show()

Analisis Versi Aplikasi vs Average Rating
---

In [ ]:
top_versions = df['versi_aplikasi'].value_counts().head(10).index
df_v = df[df['versi_aplikasi'].isin(top_versions)]
version_stats = df_v.groupby('versi_aplikasi')['rating'].mean().reset_index()

fig3 = px.scatter(version_stats, x='versi_aplikasi', y='rating', 
                 size=df_v.groupby('versi_aplikasi').size().values,
                 color='rating', color_continuous_scale='RdYlGn',
                 title='<b>Rata-rata Rating per Versi Aplikasi (Top 10)</b>',
                 labels={'rating': 'Avg Rating', 'versi_aplikasi': 'Versi'},
                 template=template)
fig3.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_20264\2695275825.py:4: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_20264\2695275825.py:7: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Distribusi Panjang Kata (Box Plot untuk Outliers
---

In [9]:
df['word_count'] = df['ulasan'].str.split().str.len()
fig4 = px.box(df[df['word_count'] < 100], y='word_count', x='rating',
             color='rating', notched=True,
             title='<b>Sebaran Panjang Kata per Rating (Hingga 100 Kata)</b>',
             template=template)
fig4.show()

Karakteristik Ulasan: Panjang Kata per Rating
---

In [10]:
df['word_count'] = df['ulasan'].astype(str).apply(lambda x: len(x.split()))
fig4 = px.violin(df[df['word_count'] < 100], y="word_count", x="rating", 
                color="rating", box=True, points="all",
                title="<b>Distribusi Panjang Kata Berdasarkan Rating</b>",
                labels={'word_count': 'Jumlah Kata', 'rating': 'Rating'},
                template=template)
fig4.show()

Insight:
****
Grafik 4 (Violin Plot): Menunjukkan perilaku user. Biasanya, user yang marah (Rating 1) menulis lebih panjang dan detail dibandingkan user yang puas (Rating 5).

Analisis Versi Aplikasi vs Kepuasan (Heatmap Style)
---

In [11]:
top_versions = df['versi_aplikasi'].value_counts().head(15).index
df_v = df[df['versi_aplikasi'].isin(top_versions)]
v_pivot = df_v.groupby(['versi_aplikasi', 'rating']).size().unstack(fill_value=0)

fig3 = px.imshow(v_pivot, 
                labels=dict(x="Rating (Bintang)", y="Versi Aplikasi", color="Jumlah"),
                title="<b>Heatmap: Versi Aplikasi vs Kualitas Rating</b>",
                color_continuous_scale='YlGnBu',
                template=template)
fig3.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_20264\427280186.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

